#### 1. Milvus 2.6 能力概览

**1.1 降本提效**

- **RaBitQ 量化索引**
  - 主索引向量 1-bit 量化，内存占用降至原来的约 1/32。
  - 叠加 SQ8 精排后，整体内存占比约 28%。
  - QPS 提升约 4 倍，召回率仍可保持在 ~95%。

- **Sparse-BM25**
  - 相比 ElasticSearch：检索速度快 3–4 倍（部分数据集可达 7 倍）。
  - 索引体积约为原始数据体积的 1/3，显著降低内存和存储成本。

- **JSON Path & JSON Shredding Index**
  - 支持对动态 JSON 字段的特定路径建立索引。
  - 先按 JSON path 过滤再向量/标量加速检索。
  - 过滤延迟从平均 140ms（P99 480ms）降至约 1.5ms（P99 10ms），适合复杂元数据过滤场景。

- **新增多种数据类型**
  - `int8` 向量：适配高性价比模型推理场景（量化后的特征向量存储）。
  - `Geometry`：支持存储/检索 POINT、LINESTRING、POLYGON 等空间形状，可用于地理围栏、导航、地图应用。
  - `Struct Array`：适合多层嵌套、属性繁多的数据建模，简化 schema 设计，增强元数据丰富场景的查询能力。

- **部分 Upsert 能力**
  - 支持仅更新记录中的部分字段，无需重写整条记录。

**1.2 搜索与功能增强**

- **文本处理升级**
  - 新增分词器：Lindera / ICU（支持日韩及多语言）。
  - Jieba 分词支持自定义词表。
  - 新增 `Run Analyzer` 语法，可对 analyzer 配置做可观测调试。
  - 支持多语言 analyzer。

- **精准匹配能力**
  - 新增 Phrase Match（短语匹配），支持 `slop` 参数控制词序宽容度，适配法律文档、智能问答等高精度场景。
  - 新增 NGRAM Index：
    - 加速 `VARCHAR` 字段的 `LIKE` 查询。
    - 也可加速 `JSON` 字段内特定 JSON path 的 `LIKE` 查询。

- **动态排序（重排）**
  - 引入 Decay Function 时间衰减重排：
    - 支持指数 / 线性 / 高斯三种衰减函数。
    - 用于提升结果的时效性（时间越近，得分越高）。
  - 引入 boost reranker：
    - 支持携带过滤条件对候选结果做匹配并加权打分，实现融合排序。

- **模型集成简化**
  - 内置集成 OpenAI、Hugging Face 等第三方模型。
  - 插入/查询时可自动将文本向量化，无需业务侧单独生成向量。

- **在线新增标量字段**
  - 支持不停机为 Collection 新增字段。
  - 无需重建 schema / 重建 Collection，直接在存量 Collection 上添加字段。

- **MinHash 能力**
  - 通过 MinHash 加速大规模文档近重复检测和相似内容识别。

**1.3 架构与存储优化**

- **冷热分层存储（Tiered Storage）**
  - 热数据存放在 SSD，冷数据下沉到对象存储。
  - 支持延迟加载和部分加载。
  - 整体资源使用几乎减半，集合加载速度明显提升。

- **实时流处理（Streaming Service）**
  - 新增 `Streaming Node`：
    - 对接 Kafka / Pulsar 等 MQ。
    - 负责实时数据接入、即时索引和查询。
    - 写入吞吐比传统 MQ 方案更高，故障恢复更快。

- **其他架构增强**
  - 支持 100k 级 Collection 数量，适合海量租户与业务隔离。
  - 升级云原生日志系统 Woodpecker（Zero-disk WAL），降低磁盘依赖。
  - Storage v2：优化 IOPS 与内存使用。
  - Coord Merge：提升集群稳定性。

---

#### 2. 2.5 → 2.6 架构组件变化

**2.1 2.5 架构问题（流批耦合）**

- Worker Nodes 中流处理与批处理耦合：
  - `QueryNode`：既负责增量数据检索，又处理历史数据查询。
  - `DataNode`：既负责增量数据 Flush 落盘，又负责历史数据 Compaction。
- 问题：
  - 批处理能力无法集中池化。
  - 流处理状态分散在不同角色，状态对齐存在延迟。

**2.2 2.6 架构优化（流批分离 + 角色合并）**

- **流批分离**
  - 新增 `StreamingNode`：
    - 专门负责流式数据的消费、写入、处理。
    - 替代 2.5 中：
      - DataNode 消费数据并落盘到对象存储。
      - QueryNode 增量数据检索。
      - Proxy 将数据写入消息流等职责。
  - `DataNode`、`QueryNode` 在 2.6 中更多聚焦批式数据处理（历史数据）。

- **角色合并**
  - 将多个 Coordinator：
    - `RootCoord` / `QueryCoord` / `DataCoord`
    - 合并为单一组件 `MixCoord`。
  - 将 `IndexNode` 和（旧）`DataNode` 能力合并为统一的 `DataNode` 组件。

总结变更要点：

- 流批分离，合并分散角色。
- 新增 `Streaming Node`。
- `MixCoord` 取代原各类 Coordinator。
- `IndexNode` 功能并入新的 `DataNode`，最终下线独立 Index Node。

---

#### 3. 2.5 → 2.6 正确升级顺序（集群）

为尽可能保持系统可用性，需要严格按以下顺序滚动升级：

1. **预先拉起 Streaming Node**
   - 将新的 Delegator（QueryNode 中负责流数据的组件）指向 2.6 Streaming Node。

2. **升级 MixCoord**
   - 2.6 的 MixCoord 需兼容识别新旧 Worker Node 版本，处理跨版本不兼容问题。

3. **升级 Query Node**
   - QueryNode 升级过程中较慢。
   - 保留 2.5 的 DataNode / IndexNode，代为承担 Flush / Index 等，减轻查询压力。

4. **升级 Data Node**
   - 注意：当 2.5 DataNode 下线后，Flush 将暂时无法完成。
   - Growing segments 数量会持续增加，直到所有 DataNode 升级到 2.6。

5. **升级 Proxy**
   - 注意：新 2.6 Proxy 上的写操作不可用，直到所有节点都升级完毕。

6. **下线 Index Node**
   - 2.6 后索引能力由 DataNode 接管。

**整体顺序：**

> 拉起 Streaming Node
> → 升级 MixCoord
> → 升级 Query Node
> → 升级 Data Node
> → 升级 Proxy
> → 下线 Index Node

**重要限制：**

- 从升级 DataNode 完成 → Proxy 全部升级完成之间：
  - **Flush 操作不可用**。
- 从第一个 Proxy 升级到 2.6 起，到所有 Proxy 升级完成：
  - **部分写操作不可用**。
- 若从 2.5.x 直接升级到 **2.6.6**：
  - 因 DDL 框架变更，升级过程中 **DDL 操作不可用**。

---

#### 4. 使用 Milvus-Operator 的升级控制（集群模式）

**4.1 Milvus-Operator 简述**

- 仓库：`github.com/zilliztech/milvus-operator`
- 用途：
  - 在 Kubernetes 中以可扩展、高可用方式部署和管理 Milvus 服务栈。
  - 管理 Milvus 组件及依赖（etcd、Pulsar、MinIO 等）。
- 实现方式：
  - 定义 `Milvus` CRD。
  - 使用 Operator 模式持续对比实际状态与期望状态并对齐。

**4.2 Milvus CR 示例（核心字段）**

```yaml
apiVersion: milvus.io/v1beta1
kind: Milvus
metadata:
  name: my-milvus-mansion
  namespace: dev
spec:
  mode: cluster    # cluster or standalone
  components:
    image: milvusdb/milvus:v2.6.5
    imageUpdateMode: rollingUpgrade
    proxy:
      replicas: 1
    mixCoord:
      replicas: 1
    dataNode:
      replicas: 1
    queryNode:
      replicas: 2
      resources:
        requests:
          cpu: "2"
          memory: "8Gi"
  dependencies:
    etcd:
      inCluster:
        values:
          replicaCount: 3
    storage:
      type: MinIO
      inCluster:
        values:
          mode: distributed
    msgStreamType: pulsar
    pulsar:
      inCluster:
        values:
          bookkeeper:
            replicas: 3
  config:
    dataCoord:
      enableActiveStandby: true
```

**4.3 Operator 对 2.5 → 2.6 滚动升级的适配逻辑**

- 版本识别：
  - 通过 `spec.components.image` 的 tag 判断是否为 2.6。
  - 或通过 `spec.components.version` 显式指定版本。
- 升级场景识别：
  - 对比 `status.currentImage` / `status.currentVersion`
  - 与 `spec.components.image` / `spec.components.version`，
  - 判断是否处于 2.5 → 2.6 的特殊升级场景。
- 在滚动升级模式 (`spec.components.imageUpdateMode: rollingUpgrade`，默认) 下：
  - 按以下顺序自动执行：
    - 拉起 `Streaming Node`
    - 升级 `MixCoord`
    - 升级 `Query Node`
    - 升级 `Data Node`
    - 升级 `Proxy`
    - 下掉 `Index Node`
- Coords 合并处理：
  - 如果 `spec.components.mixCoord` 有配置：
    - 在 MixCoord 启动后，Operator 会自动下线其他 Coordinator（Root/Query/DataCoordinators）。

**4.4 使用 Operator 的具体升级步骤示例**

a. **升级 Milvus-Operator 本身**（例：升级到 v1.3.3）

```bash
# Option 1: 使用 Helm
helm upgrade --install milvus-operator \
  -n milvus-operator --create-namespace \
  https://github.com/zilliztech/milvus-operator/releases/download/v1.3.3/milvus-operator-1.3.3.tgz

# Option 2: 使用 kubectl + 原生 manifests
kubectl apply -f https://raw.githubusercontent.com/zilliztech/milvus-operator/v1.3.3/deploy/manifests/deployment.yaml
```

b. **合并 Coords 为 MixCoord**（若已是 MixCoord 部署可跳过）

```bash
kubectl patch milvus my-release -n demo-operator \
  --type=merge -p '{
    "spec": {
      "components": {
        "mixCoord": {
          "replicas": 1
        }
      }
    }
  }'
```

c. **先统一到 2.5.16+（例：2.5.22），确保具备升级前提**

```bash
kubectl patch milvus my-release -n demo-operator \
  --type=merge -p '{
    "spec": {
      "components": {
        "image": "milvusdb/milvus:v2.5.22"
      }
    }
  }'

# 等待升级完成
kubectl wait milvus my-release -n demo-operator \
  --for=condition=milvusupdated --timeout=1h
```

d. **再升级到 2.6（例：2.6.5）**

```bash
kubectl patch milvus my-release -n demo-operator \
  --type=merge -p '{
    "spec": {
      "components": {
        "image": "milvusdb/milvus:v2.6.5"
      }
    }
  }'

# 等待升级完成
kubectl wait milvus my-release -n demo-operator \
  --for=condition=milvusupdated --timeout=1h
```

---

#### 5. 使用 Helm 的升级控制（集群模式）

> 注意：Helm 中各 Deployment 默认并发更新，没有严格顺序控制；生产环境推荐使用 Milvus-Operator。

**前置要求**

- Helm ≥ 3.14.0
- Kubernetes ≥ 1.20.0

**Step 1：升级 Helm Chart 到最新版本（示例：5.0.7）**

```bash
helm repo add zilliztech https://zilliztech.github.io/milvus-helm
helm repo update
```

**Step 2：在多 Coords 场景下，先统一到 2.5.16+ 并启用 mixCoordinator**

```bash
helm upgrade -i my-release zilliztech/milvus \
  --namespace=helm-demo \
  --set image.all.tag="v2.5.22" \
  --set mixCoordinator.enabled=true \
  --set rootCoordinator.enabled=false \
  --set indexCoordinator.enabled=false \
  --set queryCoordinator.enabled=false \
  --set dataCoordinator.enabled=false \
  --set streaming.enabled=false \
  --set indexNode.enabled=true \
  --reset-then-reuse-values \
  --version=5.0.7 \
  --wait --timeout 1h
```

**Step 3：升级到 2.6（启用 Streaming，停用 IndexNode）**

```bash
helm upgrade my-release zilliztech/milvus \
  --namespace=helm-demo \
  --set image.all.tag="v2.6.5" \
  --set streaming.enabled=true \
  --set indexNode.enabled=false \
  --reset-then-reuse-values \
  --version=5.0.7 \
  --wait --timeout 1h
```

---

#### 6. 常见问题要点

- **Helm vs Milvus-Operator**
  - 生产环境优先推荐 Milvus-Operator（具备顺序控制、自动识别升级场景等）。
  - 对比细节见 Milvus-Operator 仓库 Readme。

- **Message Queue 选择**
  - standalone：
    - 成本敏感可用 `RocksMQ`。
  - cluster：
    - `Pulsar`：支持多租户、大规模实例共享，扩缩容弹性好。
    - `Kafka`：部署维护成熟，各大云厂商提供托管服务。
  - **2.6 新的 Woodpecker：**
    - 零外部 MQ 依赖，成本低、维护简单。
    - 当前支持嵌入模式，较轻量。
    - 推荐：
      - 2.6 standalone：优先用 Woodpecker。
      - 2.6 cluster 生产：建议等待 Woodpecker 集群模式发布后使用。

- **升级过程中是否支持 MQ 切换？**
  - 当前不支持在升级过程中切换 MQ。
  - 规划通过管理 API 支持 Pulsar / Kafka / Woodpecker / RocksMQ 之间的平滑切换。

- **2.6 限流配置是否需要调整？**
  - 不需要。
  - 现有限流配置会自动对 StreamingNode 生效。

- **MixCoord 合并后的监控与配置变化？**
  - 监控 role 名称仍沿用原有：`rootcoord`, `querycoord`, `datacoord`。
  - 配置项不变，但新增：
    - `mixCoord.enableActiveStandby`
    - 若未配置，会回退使用 `rootcoord.enableActiveStandby`。

- **Streaming Node 推荐资源规格？**
  - 少量实时写入或轻量“边写边查”场景：约 `2C 8G` 即可。
  - 高吞吐实时写入/查询：可参考与 QueryNode 同等级配置。

- **Docker Compose 单机升级**
  - 修改 `docker-compose.yaml` 中 Milvus 镜像 tag 即可完成版本升级。
  - 具体参考官方 standalone Docker 升级文档。

---

#### 7. 官方升级文档参考

- 集群 + Operator：`upgrade_milvus_cluster-operator.md`
- standalone + Operator：`upgrade_milvus_standalone-operator.md`
- 集群 + Helm：`upgrade_milvus_cluster-helm.md`
- standalone + Helm：`upgrade_milvus_standalone-helm.md`